Lab 1. Fake news detection (binary classification)

In [15]:
import pandas as pd

df = pd.read_csv('/Users/e.baronov/Programming/Materials/laboratory works/lab_1/fake_or_real_news.csv')
df.head(15)

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL
5,6903,"Tehran, USA","\nI’m not an immigrant, but my grandparents ...",FAKE
6,7341,Girl Horrified At What She Watches Boyfriend D...,"Share This Baylee Luciani (left), Screenshot o...",FAKE
7,95,‘Britain’s Schindler’ Dies at 106,A Czech stockbroker who saved more than 650 Je...,REAL
8,4869,Fact check: Trump and Clinton at the 'commande...,Hillary Clinton and Donald Trump made some ina...,REAL
9,2909,Iran reportedly makes new push for uranium con...,Iranian negotiators reportedly have made a las...,REAL


# EDA

## Dataset overview: check size, number of classes, balance, average text length.

In [18]:
# size
df.shape

(6335, 4)

In [22]:
# number of classes and balance
df['label'].value_counts()

label
REAL    3171
FAKE    3164
Name: count, dtype: int64

In [23]:
df['label'].value_counts(normalize=True)

label
REAL    0.500552
FAKE    0.499448
Name: proportion, dtype: float64

In [19]:
print(3171 + 3164) # d

6335


In [82]:
def avg_statistics(series: pd.Series):
    series_name = series.name
    awc = series.str.split().str.len().mean()
    asl = series.str.len().mean()
    awl = (asl - awc + 1) / awc

    return (
        (f'average words count in {series_name}', awc),
        (f'average symbols length in {series_name}', asl),
        (f'average words length in {series_name}', awl)
    )

In [83]:
avg_statistics(df['title'])

(('average words count in title', np.float64(10.496448303078138)),
 ('average symbols length in title', np.float64(65.2776637726914)),
 ('average words length in title', np.float64(5.314294307842695)))

In [84]:
avg_statistics(df['text'])


(('average words count in text', np.float64(776.3007103393844)),
 ('average symbols length in text', np.float64(4707.250355169692)),
 ('average words length in text', np.float64(5.064982670325436)))

In [85]:
# average words count in title
awc_title = df['title'].str.split().str.len().mean()
print(awc_title)

10.496448303078138


In [58]:
# average symbols length in title
asl_title = df['title'].str.len().mean()
print(asl_title)

65.2776637726914


In [59]:
# average words length in title
awl_title = (asl_title - awc_title + 1) / awc_title
print(awl_title)

5.314294307842695


# average words count
awc = df['title'].str.split().str.len().mean()
print(awc)тоже самое для text

In [60]:
# average words count in text
awc_text = df['text'].str.split().str.len().mean()
print(awc_text)

776.3007103393844


In [63]:
# average symbols length in text
asl_text = df['text'].str.len().mean()
print(asl_text)

4707.250355169692


In [64]:
# average words length in text
awl_text = (asl_text - awc_text + 1) / awc_text
print(awl_text)

5.064982670325436


Может быть, есть смысл сравнить эти показатели у разных маркировок

In [86]:
fake_df = df[df['label'] == 'FAKE'].drop('label', axis=1)
real_df = df[df['label'] == 'REAL'].drop('label', axis=1)

In [95]:
fake_title_st = avg_statistics(fake_df['title'])

In [96]:
fake_text_st = avg_statistics(fake_df['text'])

In [97]:
real_title_st = avg_statistics(real_df['title'])

In [98]:
real_text_st = avg_statistics(real_df['text'])

In [99]:
import pandas as pd

index_names = [
    fake_title_st[0][0],
    fake_title_st[1][0],
    fake_title_st[2][0],
    fake_text_st[0][0],
    fake_text_st[1][0],
    fake_text_st[2][0],
]

df = pd.DataFrame(index=index_names, columns=['fake', 'real'])

df.loc[fake_title_st[0][0], 'fake'] = round(float(fake_title_st[0][1]), 2)
df.loc[fake_title_st[1][0], 'fake'] = round(float(fake_title_st[1][1]), 2)
df.loc[fake_title_st[2][0], 'fake'] = round(float(fake_title_st[2][1]), 2)

df.loc[fake_text_st[0][0], 'fake'] = round(float(fake_text_st[0][1]), 2)
df.loc[fake_text_st[1][0], 'fake'] = round(float(fake_text_st[1][1]), 2)
df.loc[fake_text_st[2][0], 'fake'] = round(float(fake_text_st[2][1]), 2)

df.loc[real_title_st[0][0], 'real'] = round(float(real_title_st[0][1]), 2)
df.loc[real_title_st[1][0], 'real'] = round(float(real_title_st[1][1]), 2)
df.loc[real_title_st[2][0], 'real'] = round(float(real_title_st[2][1]), 2)

df.loc[real_text_st[0][0], 'real'] = round(float(real_text_st[0][1]), 2)
df.loc[real_text_st[1][0], 'real'] = round(float(real_text_st[1][1]), 2)
df.loc[real_text_st[2][0], 'real'] = round(float(real_text_st[2][1]), 2)


print(df)

                                    fake     real
average words count in title       11.13     9.86
average symbols length in title    69.18    61.38
average words length in title        5.3     5.33
average words count in text       679.13   873.26
average symbols length in text   4121.05  5292.16
average words length in text        5.07     5.06


In [101]:
import pandas as pd

df = pd.DataFrame(index=[fake_title_st[i][0] for i in range(3)] +
                         [fake_text_st[i][0] for i in range(3)],
                  columns=['fake', 'real'])

for ft, rt in zip(fake_title_st, real_title_st):
    df.loc[ft[0], 'fake'] = round(float(ft[1]), 2)
    df.loc[rt[0], 'real'] = round(float(rt[1]), 2)

for ft, rt in zip(fake_text_st, real_text_st):
    df.loc[ft[0], 'fake'] = round(float(ft[1]), 2)
    df.loc[rt[0], 'real'] = round(float(rt[1]), 2)

print(df)

                                    fake     real
average words count in title       11.13     9.86
average symbols length in title    69.18    61.38
average words length in title        5.3     5.33
average words count in text       679.13   873.26
average symbols length in text   4121.05  5292.16
average words length in text        5.07     5.06


При практически одинаковой длине слова мы видим разницу в 12 % между количеством слов и символов( у fake больше), при этом есть разница около 22% у количества слов и символов (у real больше).  отсюда можем сделать вывод что у фейковых новостей немного длиннее заголовки, но практически на одну треть короче текст новости чем у настоящих

In [102]:
words_title_diff = ((11.13 - 9.86) / 9.86) * 100
symbols_title_diff = ((69.18 - 61.38) / 61.38) * 100
wordlen_title_diff = ((5.30 - 5.33) / 5.33) * 100

words_text_diff = ((679.13 - 873.26) / 873.26) * 100
symbols_text_diff = ((4121.05 - 5292.16) / 5292.16) * 100
wordlen_text_diff = ((5.07 - 5.06) / 5.06) * 100

In [103]:
print(words_title_diff)
print(symbols_title_diff)
print(wordlen_title_diff)
print(words_text_diff)
print(symbols_text_diff)
print(wordlen_text_diff)

12.880324543610563
12.707722385141745
-0.5628517823639821
-22.230492636786295
-22.129149534405606
0.19762845849803706
